In [0]:
%run ../../Include/DatabricksHelpers

In [0]:
%run ../../Include/PipelineTests

We load and instantiate DatabricksHelpers to discover the active user's identity and derive an isolated exercise working directory. We then ensure the target catalog schema (f1_dev.bronze) is created before running any ingestion.

In [0]:
helpers = DatabricksHelpers(dbutils)

current_user = helpers.current_user()
project_root = helpers.project_root()

print(f"Running pipeline as: {current_user}")
print(f"Project Root:         {project_root}")

We initialize our catalog targets and use helpers to resolve the current user and establish our exercise workspace. We also ensure the bronze schema exists inside Unity Catalog before writing any data.

In [0]:
catalog_name = "f1_dev"
schema_name = "bronze"

# Ensure target Bronze schema exists in catalog
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")

In [0]:
# ASSERTION: Verify schema exists in target catalog
schemas_df = spark.sql(f"SHOW SCHEMAS IN {catalog_name}")
schema_list = [row["databaseName"] for row in schemas_df.collect()]

assert schema_name in schema_list, f"❌ TEST FAILED: Schema '{schema_name}' was not found in catalog '{catalog_name}'!"

print(f"✓ TEST PASSED: Target schema '{catalog_name}.{schema_name}' verified and accessible.")

Instead of hardcoding personal user paths (/Users/userName@...), we resolve the raw dataset location dynamically relative to your workspace root or helper directory. We test file existence before triggering any Spark reads.

In [0]:
import os

# 1. Get current notebook's workspace folder
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
notebook_dir = os.path.dirname(f"/Workspace{notebook_path}")

# 3. Construct dynamic path to raw data
raw_file_path = os.path.join(project_root, "data/raceHistoricalData.csv")

# 4. Format path with file: prefix for PySpark compatibility
spark_file_path = f"file:{raw_file_path}"

In [0]:
# ASSERTION: Verify physical file exists on workspace disk
assert os.path.exists(raw_file_path), f"❌ TEST FAILED: Source CSV missing at '{raw_file_path}'"

print(f"✓ TEST PASSED: File resolved at '{raw_file_path}'")

We read the raw CSV directly into a PySpark DataFrame using spark_file_path (file:/Workspace/...). Reading natively distributes file processing across cluster workers, avoiding driver node memory bottlenecks caused by Pandas (pd.read_csv). We then immediately assert that the DataFrame is populated and adheres to the expected column schema.

In [0]:
df_bronze = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(spark_file_path)
)

In [0]:
raw_count = test_dataframe_not_empty(df_bronze, "df_bronze")
test_columns_exist(df_bronze, ["Date", "Open", "High", "Low", "Close/Last", "Volume"], "df_bronze")

We persist the raw PySpark DataFrame into Unity Catalog as a Bronze layer Delta table. Using overwrite mode and overwriteSchema ensures that the table creation is idempotent, allowing any team member to re-run the cell cleanly without duplicating records or failing on schema conflicts.

In [0]:
target_table_name = "ferrari_stock_raw"
full_target_table = f"{catalog_name}.{schema_name}.{target_table_name}"

(
    df_bronze.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(full_target_table)
)

In [0]:
test_table_exists_and_row_count(spark, catalog_name, schema_name, target_table_name, raw_count)

To enforce enterprise data governance standards, we attach metadata tags to our newly created Bronze Delta table using Spark SQL. These tags classify the dataset as public financial data without PII, tracking its business metadata (ticker RACE) directly in Unity Catalog.

In [0]:
spark.sql(f"""
    ALTER TABLE {target_table} SET TAGS (
        'classification' = 'Public_Financial',
        'contains_pii' = 'false',
        'ticker' = 'RACE'
    )
""")

In [0]:
expected_governance = {
    "classification": "Public_Financial",
    "contains_pii": "false",
    "ticker": "RACE"
}

# Reusable test from Block 0
test_table_tags(spark, catalog_name, schema_name, target_table_name, expected_governance)

print("🎉 BRONZE LAYER INGESTION PIPELINE FULLY VERIFIED AND COMPLETE!")

Summary Check

Workspace Cleanup: Operates strictly on temporary local files in /Workspace.

Delta Table Persistence: Standard Unity Catalog managed tables write raw underlying Delta files to cloud object storage.

If you drop the table via SQL (DROP TABLE f1_dev.bronze.ferrari_stock_raw), the data files in cloud storage drop with it. But cleaning your local workspace folder via helpers.clean_working_directory() will never touch your Bronze Delta tables.